In [1]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement, BatchStatement
from cassandra import ConsistencyLevel
import datetime

cluster = Cluster(['localhost'], port=9042)
session = cluster.connect('leaderboards')


In [2]:
import pandas as pd

## Lecturas

### Hall of Fame

In [3]:
paises = pd.read_csv('./csv_tablas/dungeons_by_country.csv')

In [4]:
paises.country.unique()

<StringArray>
['ja_JP', 'en_US', 'fr_FR', 'ko_KR', 'pt_BR', 'it_IT', 'es_ES', 'de_DE',
 'ru_RU', 'zh_CN', 'zh_TW']
Length: 11, dtype: str

In [5]:
def id_dungeons_of_country(session, country):
    query = "SELECT dungeon_id FROM dungeons_by_country WHERE country = %s;"
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [country])
    
    dungeons_id = []

    for fila in resultados:
        dungeons_id.append(fila.dungeon_id)

    return dungeons_id

In [6]:
id_dungeons_of_country(session, 'es_ES')

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [7]:
def top_by_dungeon_and_country(session, country, dungeon_id, k=5):
    query = "SELECT dungeon_id, dungeon_name, time_minutes, user_name, email, date FROM hall_of_fame_by_country WHERE country = %s AND dungeon_id = %s LIMIT %s;"
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [country, dungeon_id, k])
    
    dungeon_id = resultados[0].dungeon_id
    dungeon_name = resultados[0].dungeon_name

    top = []

    for fila in resultados:
        top.append({'email': fila.email, 'user_name': fila.user_name, 'time_minutes': fila.time_minutes, 'date': fila.date.isoformat()})

    return {'dungeon_id': dungeon_id, 'dungeon_name': dungeon_name, f'top_{k}': top}


In [8]:
top_by_dungeon_and_country(session, 'es_ES', 1, 5)

{'dungeon_id': 1,
 'dungeon_name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
 'top_5': [{'email': 'abellanjulio@example.org',
   'user_name': 'garciamartirio',
   'time_minutes': 0,
   'date': '2018-12-04T02:45:50'},
  {'email': 'agulloricarda@example.org',
   'user_name': 'angelino53',
   'time_minutes': 0,
   'date': '2012-10-08T01:21:09'},
  {'email': 'amadorbarbero@example.com',
   'user_name': 'sbarba',
   'time_minutes': 0,
   'date': '2022-10-10T03:12:30'},
  {'email': 'berta74@example.com',
   'user_name': 'aparicioromulo',
   'time_minutes': 0,
   'date': '2016-09-26T04:29:39'},
  {'email': 'brionesjose-antonio@example.com',
   'user_name': 'brumaricruz',
   'time_minutes': 0,
   'date': '2022-03-11T00:22:35'}]}

In [9]:
def hall_of_fame(session, country):
    
    dungeon_ids = id_dungeons_of_country(session, country)

    tops_pais = []

    for dungeon_id in dungeon_ids:
        tops_pais.append(top_by_dungeon_and_country(session, country, dungeon_id, 5))
        
    return tops_pais


In [10]:
hall_of_fame(session, 'es_ES')

### User Statistics 

In [11]:
def user_statistics(session, email, dungeon_id):
    query = """SELECT time_minutes, date 
                FROM user_statistics_by_dungeon 
                WHERE email = %s AND dungeon_id = %s;"""

    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [email, dungeon_id])

    stats = []

    for fila in resultados:
        stats.append({'time_minutes': fila.time_minutes, 'date': fila.date.isoformat()})

    return stats

In [12]:
user_statistics(session, 'aabe@example.net', 0)

[{'time_minutes': 6, 'date': '2022-04-11T10:43:14'},
 {'time_minutes': 19, 'date': '2021-01-18T18:50:53'},
 {'time_minutes': 20, 'date': '2020-04-18T04:46:20'},
 {'time_minutes': 20, 'date': '2021-06-21T05:53:27'},
 {'time_minutes': 25, 'date': '2021-11-03T11:29:36'},
 {'time_minutes': 26, 'date': '2022-07-17T15:34:08'},
 {'time_minutes': 27, 'date': '2022-06-05T04:55:02'},
 {'time_minutes': 35, 'date': '2022-07-13T18:17:30'},
 {'time_minutes': 38, 'date': '2020-09-06T11:36:08'},
 {'time_minutes': 40, 'date': '2020-12-05T08:54:05'},
 {'time_minutes': 40, 'date': '2022-05-09T12:28:38'},
 {'time_minutes': 43, 'date': '2022-09-12T23:54:05'}]

### Top Horde

In [13]:
def top_horde(session, country, event_id, K):

    query = """SELECT email, user_name, n_killed
                FROM top_horde_by_event 
                WHERE country = %s AND event_id = %s 
                LIMIT %s;"""
    
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.ONE)

    resultados = session.execute(statement, [country, event_id, K])

    top = []

    for fila in resultados:
        top.append({'email': fila.email, 'user_name': fila.user_name, 'n_killed': fila.n_killed})

    return top

In [14]:
top_horde(session, 'de_DE', 0, 5)

[{'email': 'xullrich@example.org',
  'user_name': 'hartmanngloria',
  'n_killed': 19},
 {'email': 'silvesterconradi@example.net',
  'user_name': 'bernhardmangold',
  'n_killed': 15},
 {'email': 'gretelholsten@example.com',
  'user_name': 'johannekruschwitz',
  'n_killed': 14},
 {'email': 'pero13@example.net', 'user_name': 'zdravko34', 'n_killed': 14},
 {'email': 'roehrichtpaulina@example.org',
  'user_name': 'oderwaldgabriella',
  'n_killed': 14}]

## Escritura

### User finish dungeon

In [ ]:
def user_finish_dungeon(session, country, dungeon_id, time_minutes, email, user_name, date, dungeon_name):
    batch = BatchStatement(consistency_level=ConsistencyLevel.QUORUM)
    
    # Registrar la combinación país-mazmorra (idempotente)
    insert_dungeons_by_country = """
        INSERT INTO dungeons_by_country 
        (country, dungeon_id) 
        VALUES (%s, %s)
    """
    
    # Insertar en el Hall of Fame
    insert_hall_of_fame = """
        INSERT INTO hall_of_fame_by_country 
        (country, dungeon_id, time_minutes, email, user_name, date, dungeon_name) 
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """

    # Insertar en las estadísticas personales del usuario
    insert_user_stats = """
        INSERT INTO user_statistics_by_dungeon 
        (email, dungeon_id, time_minutes, date) 
        VALUES (%s, %s, %s, %s)
    """
    
    batch.add(insert_dungeons_by_country, (country, dungeon_id))
    batch.add(insert_hall_of_fame, (country, dungeon_id, time_minutes, email, user_name, date, dungeon_name))
    batch.add(insert_user_stats, (email, dungeon_id, time_minutes, date))
    

    session.execute(batch)
    print(f"Récord de {user_name} guardado con éxito en las tres tablas.")

In [ ]:
user_finish_dungeon(
    session=session,
    country='ES',
    dungeon_id=101,
    time_minutes=11,
    email='player@email.com',
    user_name='Thor',
    date=datetime.datetime.now().isoformat(),
    dungeon_name='Cueva Helada'
)

Récord de Thor guardado con éxito en las tres tablas.


In [26]:
user_statistics(session, 'player@email.com', 101)

[{'time_minutes': 11, 'date': '2026-03-16T14:06:11.014000'},
 {'time_minutes': 11, 'date': '2026-03-16T16:30:00.229000'},
 {'time_minutes': 11, 'date': '2026-03-16T16:40:27.793000'},
 {'time_minutes': 11, 'date': '2026-03-16T16:40:29.893000'},
 {'time_minutes': 12, 'date': '2026-03-16T16:40:34.450000'}]

In [27]:
hall_of_fame(session, 'ES')

[{'dungeon_id': 101,
  'dungeon_name': 'Cueva Helada',
  'top_5': [{'email': 'player@email.com',
    'user_name': 'Thor',
    'time_minutes': 11,
    'date': '2026-03-16T16:40:29.893000'},
   {'email': 'player@email.com',
    'user_name': 'Thor',
    'time_minutes': 12,
    'date': '2026-03-16T16:40:34.450000'}]}]

### User kills monster during Horde event

In [32]:
def user_kills_monster(session, country, event_id, email, user_name):
    # Paso 1: Leer n_killed actual del jugador (usa el índice secundario sobre email)
    get_kills_query = """
        SELECT n_killed
        FROM top_horde_by_event
        WHERE country = %s AND event_id = %s AND email = %s;
    """
    statement = SimpleStatement(get_kills_query, consistency_level=ConsistencyLevel.ONE)
    row = session.execute(statement, [country, event_id, email]).one()

    # Paso 2: Calcular nuevo n_killed
    n_killed_actual = row.n_killed if row else 0
    n_killed_total = n_killed_actual + 1

    # Paso 3: DELETE + INSERT en BATCH
    batch = BatchStatement(consistency_level=ConsistencyLevel.ONE)

    # Borrar fila anterior si existe
    if row:
        delete_query = """
            DELETE FROM top_horde_by_event
            WHERE country = %s AND event_id = %s AND n_killed = %s AND email = %s;
        """
        batch.add(delete_query, (country, event_id, n_killed_actual, email))

    # Insertar fila actualizada
    insert_query = """
        INSERT INTO top_horde_by_event
        (country, event_id, n_killed, email, user_name)
        VALUES (%s, %s, %s, %s, %s)
    """
    batch.add(insert_query, (country, event_id, n_killed_total, email, user_name))

    # Ejecutar cambios
    try:
        session.execute(batch)
        print(f"Récord de {user_name} actualizado. Total bajas: {n_killed_total}.")
    except Exception as e:
        print(f"Error al actualizar el Leaderboard: {e}")


**Nota**: Aunque el índice secundario soluciona este problema para la práctica, en sistemas masivos a nivel de empresas como Netflix o Discord, a veces prefieren crear dos tablas separadas: una tabla rápida para buscar al usuario por email y otra tabla ordenada para el leaderboard. Pero para este prototipo, el índice secundario cumple perfectamente el objetivo sin complicar el modelo de datos.

Como ejemplo vamos a modificar la siguiente fila: 

de_DE,0,7,trommleremilie,kabuskarin@example.net

In [34]:
user_kills_monster(
    session=session,
    country='de_DE',
    event_id=0,
    email='kabuskarin@example.net',
    user_name='trommleremilie'
)

Récord de trommleremilie actualizado. Total bajas: 12.
